In [14]:
# import WinoTran_NHWC as NHWC
import math
import struct
import numpy as np
import torch.nn.functional as F
import WinoTran_CHWN as CHWN

In [2]:
inside = 224
padding =1
chn =3
bat4Conv =1
numOfFilter =64

inside_beta = math.ceil((inside+2*padding-2)/4)*4+2 
blockn = (int)((inside_beta-2)/4)
M = (int)(blockn * blockn * bat4Conv);
K = chn
N = numOfFilter
NSize = (int((N-1)/128)+1)*128
MSize = M if (M%128 == 0) else math.ceil(M/128)*128
KSize = (int((K-1)/8)+1)*8

parameters1 = 36*MSize*NSize

# readin the feature map
# src = open("gemm1.bin","rb")
# context = src.read(parameters1*4)
# real_context = struct.unpack(str(parameters1)+'f',context)
# gemm1 = np.array(real_context)

In [70]:
print(M,MSize)

31360 31360


In [79]:
# 在notebook的第一个cell中运行
import resource
import sys

# 设置最大内存限制（以字节为单位）
# 例如设置为32GB
rsrc = resource.RLIMIT_AS
resource.setrlimit(rsrc, (32*1024*1024*1024, -1))

In [15]:
feature_NCHW = np.memmap('/home/wangq/Preprocessing/DatasetPreprocessing/binImage/Batch1/bat1_0.bin', 
                 dtype='float32',  # 根据实际数据类型调整
                 mode='r',         # 'r'只读，'r+'读写
                 offset=0,         # 开始读取的位置
                 shape=(bat4Conv,chn,inside,inside) # 根据实际数据维度调整
                )

feature_NHWC = np.memmap('/home/wangq/Preprocessing/DatasetPreprocessing/binImage/Batch1_NHWC/bat1_0.bin', 
                 dtype='float32',  # 根据实际数据类型调整
                 mode='r',         # 'r'只读，'r+'读写
                 offset=0,         # 开始读取的位置
                 shape=(bat4Conv,inside,inside,chn) # 根据实际数据维度调整
                )
feature_CHWN = np.memmap('/home/wangq/Preprocessing/DatasetPreprocessing/binImage/Batch1_CHWN/bat1_0.bin', 
                 dtype='float32',  # 根据实际数据类型调整
                 mode='r',         # 'r'只读，'r+'读写
                 offset=0,         # 开始读取的位置
                 shape=(chn,inside,inside,bat4Conv) # 根据实际数据维度调整
                )


weight = np.memmap('/home/wangq/Preprocessing/ModelPreprocessing/vggData/conv1.bin', 
                 dtype='float32',  # 根据实际数据类型调整
                 mode='r',         # 'r'只读，'r+'读写
                 offset=0,         # 开始读取的位置
                 shape=(numOfFilter,chn,3,3) # 根据实际数据维度调整
                )
weight_CHWN = np.memmap('/home/wangq/Preprocessing/ModelPreprocessing/vggData_CHWN2/conv1.bin', 
                 dtype='float32',  # 根据实际数据类型调整
                 mode='r',         # 'r'只读，'r+'读写
                 offset=0,         # 开始读取的位置
                 shape=(chn,3,3,numOfFilter) # 根据实际数据维度调整
                )
conv1 = np.memmap('cudnnConv_out1.bin', 
                 dtype='float32',  # 根据实际数据类型调整
                 mode='r',         # 'r'只读，'r+'读写
                 offset=0,         # 开始读取的位置
                 shape=(bat4Conv,numOfFilter,inside,inside) # 根据实际数据维度调整
                )
conv2 = np.memmap('cudnnConv_out2.bin', 
                 dtype='float32',  # 根据实际数据类型调整
                 mode='r',         # 'r'只读，'r+'读写
                 offset=0,         # 开始读取的位置
                 shape=(bat4Conv,inside,inside,numOfFilter) # 根据实际数据维度调整
                )


In [16]:
print("feature_NCHW shape:"+str(feature_NCHW.shape))
print("feature_NHWC shape:"+str(feature_NHWC.shape))
print("feature_CHWN shape:"+str(feature_CHWN.shape))
print("weight_NCHW shape:"+str(weight.shape))
print("weight_CHWN shape:"+str(weight_CHWN.shape))

feature_NCHW shape:(1, 3, 224, 224)
feature_NHWC shape:(1, 224, 224, 3)
feature_CHWN shape:(3, 224, 224, 1)
weight_NCHW shape:(64, 3, 3, 3)
weight_CHWN shape:(3, 3, 3, 64)


In [10]:
print(np.sum(feature_NCHW))
print(np.sum(feature_NHWC))
print(np.sum(feature_CHWN))

-718.53345
-718.5332
-718.53345


In [18]:
print(weight[0,0,:,:])
print(weight_CHWN[0,:,:,0])

[[-0.5537306   0.1427047   0.5289615 ]
 [-0.58312404  0.35655147  0.76566225]
 [-0.69022113 -0.04801885  0.48409155]]
[[-0.5537306   0.1427047   0.5289615 ]
 [-0.58312404  0.35655147  0.76566225]
 [-0.69022113 -0.04801885  0.48409155]]


In [20]:
import torch
import torch.nn.functional as F

def conv2d_torch(input_array, weights):
    # 转换为 torch tensor
    input_tensor = torch.from_numpy(input_array)
    weights_tensor = torch.from_numpy(weights)
    
    # 执行卷积
    output = F.conv2d(input_tensor, weights_tensor,padding=1)
    
    # 转回 numpy
    return output.numpy()
rst_NCHW = conv2d_torch(feature_NCHW,weight)
rst_CHWN = CHWN.Conv_CHWN(feature_CHWN,weight_CHWN,padding=1)

In [35]:
print(rst_NCHW.shape)
print(rst_CHWN.shape)
print(rst_CHWN_py.shape)
err = rst_NCHW - rst_CHWN.transpose([3,0,1,2])
err2 = rst_CHWN - rst_CHWN_py
print(np.sum(err))
print(np.sum(err2))

(1, 64, 224, 224)
(64, 224, 224, 1)
(64, 224, 224, 1)
0.00052333256
-0.0026641958


In [44]:
print(np.sum(rst_CHWN))
print(np.sum(rst_CHWN_py))
print(np.sum(conv_CHWN_cuda))
print(np.sum(x1))
print(np.sum(k1))
print(np.sum(o1))


-42100.312
-42100.316
-1814831.2
-705.3263
0.07917796
-2439.2017


In [37]:
conv_CHWN_cuda = np.memmap('./conv_CHWN.bin', 
                 dtype='float32',  # 根据实际数据类型调整
                 mode='r',         # 'r'只读，'r+'读写
                 offset=0,         # 开始读取的位置
                 shape=(numOfFilter,inside,inside,bat4Conv) # 根据实际数据维度调整
                )

In [40]:
print(conv_CHWN_cuda[0,0:5,0:5,0])
print(rst_CHWN_py[0,0:5,0:5,0])


[[ -7.2157774    8.71355     -8.8424225    7.3704       0.06160507]
 [ -6.9544353    9.301137    -5.917342     9.347267    -1.1800572 ]
 [ -8.551453     8.669276   -12.349267     6.8745866    1.3887775 ]
 [ -5.2884164   10.564492     1.1595285   13.603332    -3.7564788 ]
 [  3.965107    -3.8919358    3.3393607   -4.822548     1.4926536 ]]
[[-0.7200416  -0.09245882 -0.14677478 -0.06039244 -0.09917133]
 [-1.0631613  -0.11072818 -0.14936681 -0.08267264 -0.12246193]
 [-1.130595   -0.14122112 -0.15355234 -0.079312   -0.11632155]
 [-1.2151731  -0.16434719 -0.10606343 -0.14513902 -0.14881961]
 [-1.2460567  -0.12490047 -0.04228502 -0.16016094 -0.20529383]]


In [25]:
x1 =  CHWN.Wino_inputTran(feature_CHWN, padding=1)
k1 =  CHWN.Wino_kernelTran(weight_CHWN)
print(x1.shape)
print(k1.shape)

(36, 8, 3200)
(36, 8, 128)


In [33]:
x1 =  CHWN.Wino_inputTran(feature_CHWN, padding=1)
k1 =  CHWN.Wino_kernelTran(weight_CHWN)
o1 =  CHWN.gemm(x1,k1, x1.shape[2], k1.shape[2], True)
o2 =  CHWN.Wino_OutputTran(o1, inside, bat4Conv, numOfFilter)
rst_CHWN_py = CHWN.Wino_inverseTran(o2, numOfFilter, bat4Conv, blockn, inside)

In [5]:
print("feature_NCHW shape:"+str(feature_NCHW.shape))
print("feature_NHWC shape:"+str(feature_NHWC.shape))
print("weight shape:"+str(weight.shape))
print("conv1 shape:"+str(conv1.shape))
print("conv2 shape:"+str(conv2.shape))


feature_NCHW shape:(10, 3, 224, 224)
feature_NHWC shape:(10, 224, 224, 3)
weight shape:(64, 3, 3, 3)
conv1 shape:(10, 64, 224, 224)
conv2 shape:(10, 224, 224, 64)


In [22]:
print(np.sum(np.abs(feature_NCHW)))
print(np.sum(np.abs(feature_NCHW- feature_NHWC.transpose(0,3,1,2))))
conv_rst = conv2d_torch(feature_NCHW,weight)
print(np.sum(np.abs(conv1- conv_rst)))
print(np.sum(np.abs(conv1- conv2.transpose(0,3,1,2))))

1278509.2
0.0
5.6020436
0.0


In [33]:
weight = np.memmap('/home/wangq/Preprocessing/ModelPreprocessing/vggData_NHWC_CUDNN/fc14.bin', 
                 dtype='float32',  # 根据实际数据类型调整
                 mode='r',         # 'r'只读，'r+'读写
                 offset=0,         # 开始读取的位置
                 shape=(4096,512,7,7) # 根据实际数据维度调整
                )

In [36]:
import numpy as np
import os

# 1. 读取原始文件
original = np.memmap('/home/wangq/Preprocessing/ModelPreprocessing/vggData_NHWC_CUDNN/fc14.bin', 
                     dtype='float32',
                     mode='r',
                     shape=(4096,512,7,7))

# 2. 创建临时文件
temp_file = '/home/wangq/Preprocessing/ModelPreprocessing/vggData_NHWC_CUDNN/fc14_temp.bin'
new_weight = np.memmap(temp_file,
                      dtype='float32',
                      mode='w+',
                      shape=(4096,7,7,512))  # 新的目标形状

# 3. 转置并写入
new_weight[:] = np.transpose(original, (0,2,3,1))
new_weight.flush()

# 4. 关闭 memmap 文件
del original
del new_weight

# 5. 替换原文件
os.replace(temp_file, '/home/wangq/Preprocessing/ModelPreprocessing/vggData_NHWC_CUDNN/fc14.bin')

In [15]:
import torch
import torch.nn.functional as F

def conv2d_torch(input_array, weights):
    # 转换为 torch tensor
    input_tensor = torch.from_numpy(input_array)
    weights_tensor = torch.from_numpy(weights)
    
    # 执行卷积
    output = F.conv2d(input_tensor, weights_tensor,padding=1)
    
    # 转回 numpy
    return output.numpy()

In [120]:
# gemm2MT =gemm2M.transpose(0, 2, 1)
for i in range(64):
    err = np.sum(np.abs(conv1[0,i]-conv2[0,i]))
    print(err)
# print(np.sum(conv1))
# print(np.sum(conv2))

0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0


In [96]:
print(conv1.shape)
print(conv2.shape)

(10, 64, 224, 224)
(10, 64, 224, 224)


In [116]:
print(conv1[0,14,8,0:64]-conv2[0,14,8,0:64])
# print()
print(np.sum(np.sum(np.abs(conv1[0,14,8,:]-conv2[0,14,8,:]))))

[ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
  1.0236136e-01  2.0823951e-01  1.7718644e-01  4.4417381e-04
 -6.5329097e-02  1.8381694e-01  5.8578473e-01  5.7299042e-01
  3.1749848e-01  2.2446960e-01  1.5539765e-01  1.9334549e-01
  1.5648164e-01  1.2575260e-01  5.0102569e-02  3.6457345e-02
  1.2463459e-01  1.4296293e-01  1.6741115e-01  1.5821150e-01
  1.9115390e-01  2.8929737e-01  2.5079292e-01  2.1275760e-01
  1.6924271e-01  8.6475551e-02  4.3946549e-02  3.8080245e-02
  6.8061523e-02  1.4752658e-01  1.0134238e-01  1.2209071e-01
  2.3434174e-01  1.6231480e-01  1.7990659e-01  1.7890735e-01
  1.4718294e-01  2.2454061e-01  2.4393532e-01  2.8216115e-01
  0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00]
7.1229362


In [118]:
for i in range(224):
    if (np.sum(np.abs(conv1[0,14,i,:]-conv2[0,14,i,:])) !=0):
        print(i)

8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55


In [19]:
print(gemm1.shape)
print(gemm2.shape)
print(gemm1[0:2,0:2,0:2])
print(gemm2[0:2,0:2,0:2])

(36, 3200, 128)
(36, 128, 3200)
[[[ 0.2548248  -0.0081314 ]
  [-0.01256454 -0.03749925]]

 [[-0.00328726  0.01781967]
  [-0.01746764 -0.01777412]]]
[[[ 0.2548248  -0.01256454]
  [-0.00328726 -0.01746764]]

 [[ 0.04672558  0.00070973]
  [ 0.00284336  0.0025694 ]]]


In [88]:
print(x5.shape)
err1 = np.sum(np.abs(x5[0]-data2))
err2 = np.sum(np.abs(x5[1]-data2))
err3 = np.sum(np.abs(x5[2]-data2))
print(err1, err2, err3)

(5, 10, 64, 224, 224)
0.0 0.0 0.0


In [6]:

inside = 224
padding =1
chn =64
bat4Conv =10
numOfFilter =64

inside_beta = math.ceil((inside+2*padding-2)/4)*4+2 
blockn = (int)((inside_beta-2)/4)
M = (int)(blockn * blockn * bat4Conv);
K = chn
N = numOfFilter
NSize = (int((N-1)/128)+1)*128
MSize = M if (M%128 == 0) else math.ceil(M/128)*128
KSize = (int((K-1)/8)+1)*8

parameters1 = 36*MSize*NSize

# readin the feature map
src = open("gemm1.bin","rb")
context = src.read(parameters1*4)
real_context = struct.unpack(str(parameters1)+'f',context)
gemm1 = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
gemm1 = gemm1.reshape((36,MSize,NSize)).astype(np.float32)

# readin the feature map
src = open("gemm2.bin","rb")
context = src.read(parameters1*4)
real_context = struct.unpack(str(parameters1)+'f',context)
gemm2 = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
gemm2 = gemm2.reshape((36,NSize,MSize)).astype(np.float32)

In [8]:
print(gemm1.shape)
print(gemm2.shape)
# print(gemm1[0,1,0:6,0:6])
# print(gemm2[0,1,0:6,0:6])
gemm2T = gemm2.transpose(0, 2, 1)
err = np.sum(np.abs(gemm1-gemm2T))
print(err)

(36, 31360, 128)
(36, 128, 31360)
0.0


In [71]:
sum1 = np.sum(np.abs(gemm1))
print(sum1)

0.0


In [69]:
parameters2 = bat4Conv*chn*inside*inside

# readin the feature map
src = open("conv_out1.bin","rb")
context = src.read(parameters1*4)
real_context = struct.unpack(str(parameters2)+'f',context)
conv1 = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
conv1 = conv1.reshape((bat4Conv,numOfFilter,inside,inside)).astype(np.float32)

# readin the feature map
src = open("conv_out2.bin","rb")
context = src.read(parameters1*4)
real_context = struct.unpack(str(parameters2)+'f',context)
conv2 = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
conv2 = conv2.reshape((bat4Conv,numOfFilter,inside,inside)).astype(np.float32)

In [62]:
print(conv1.shape)
print(conv2.shape)
print(conv1[0,0,0:6,0:6])
print(conv2[0,0,0:6,0:6])
# gemm2T = gemm2.transpose(0, 2, 1)
# err = np.sum(np.abs(gemm1-gemm2T))
for i in range(64):
    err = np.sum(np.abs(conv1[0,i,:,:]-conv2[0,i,:,:]))
    print(err)

(10, 64, 224, 224)
(10, 64, 224, 224)
[[-0.86339074 -0.18523371  0.7491195   1.0391266   0.7147729   0.3376276 ]
 [-1.7772572  -0.17174321  0.7555784   1.3698311   1.2182134   0.60686576]
 [-1.0610507   0.07652336  0.35876775  0.7117286   0.93075466  0.49495602]
 [-1.0490448   0.0389514   0.35850084  0.5426111   0.67963636  0.528039  ]
 [-0.9446559  -0.09300163  0.12842613  0.27719045  0.18996847  0.0592218 ]
 [-0.6715955  -0.01106948  0.15799302  0.14899957  0.03517827 -0.04741328]]
[[-1.01603854e+00 -2.92202443e-01  6.75826967e-01  9.65875149e-01
   6.39904439e-01  2.59236366e-01]
 [-1.87068689e+00 -2.29543418e-01  7.36330211e-01  1.34818339e+00
   1.20972466e+00  6.09717429e-01]
 [-1.08305192e+00  2.75918245e-02  3.14864397e-01  6.62858486e-01
   8.88738811e-01  4.69445378e-01]
 [-1.07079649e+00 -3.39746475e-05  3.30563068e-01  5.14335155e-01
   6.55755281e-01  5.08244216e-01]
 [-9.68446374e-01 -1.44904554e-01  7.34602213e-02  2.08242416e-01
   1.40220717e-01  4.35204580e-02]
 [-7.1

In [47]:
i =16
print(gemm1[0,i,0:6,0:6])
print(gemm2[0,i,0:6,0:6])

[[-0.1061887  -0.2066169  -0.2700211  -0.24872617 -0.20664868 -0.17221819]
 [-0.00778634 -0.06270888 -0.10009093 -0.1524286  -0.14496534 -0.12120605]
 [-0.04637911 -0.12074039 -0.13112827 -0.14105368 -0.16974838 -0.14569716]
 [-0.04297487 -0.14608085 -0.14307207 -0.17716655 -0.19741902 -0.20558514]
 [-0.05806501 -0.13763586 -0.13728397 -0.157532   -0.1822454  -0.18985535]
 [-0.07977192 -0.14758231 -0.11964723 -0.12665741 -0.14954418 -0.15174827]]
[[-0.1061887  -0.2066169  -0.2700211  -0.24872617 -0.20664868 -0.17221819]
 [-0.00778634 -0.06270888 -0.10009093 -0.1524286  -0.14496534 -0.12120605]
 [-0.04637911 -0.12074039 -0.13112827 -0.14105368 -0.16974838 -0.14569716]
 [-0.04297487 -0.14608085 -0.14307207 -0.17716655 -0.19741902 -0.20558514]
 [-0.05806501 -0.13763586 -0.13728397 -0.157532   -0.1822454  -0.18985535]
 [-0.07977192 -0.14758231 -0.11964723 -0.12665741 -0.14954418 -0.15174827]]
